<a href="https://colab.research.google.com/github/Kishanmvs/Machine-Learning-II-Lab-BCI702/blob/main/Lab3_FindS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Find-S Algorithm — Play Tennis Dataset

**Goal:**
1. Use the **Find-S algorithm** to find the most specific hypothesis consistent with the positive examples.
2. State the final hypothesis after processing all positive examples.

**Dataset:** `play_tennis_50.csv` — attributes `Outlook`, `Temperature`, `Humidity`, `Wind`,
and target `PlayTennis` (Yes/No).

**Find-S recap:**
- Start with the most specific hypothesis: `h = <∅, ∅, ∅, ∅>` (nothing satisfies it).
- For each **positive** training example only:
  - If an attribute value in `h` differs from the example's value, generalize it to `?`.
  - If `h` is still `∅` (first positive example), initialize it directly from that example.
- Negative examples are ignored (Find-S never generalizes/checks against them).
- The result is the maximally specific hypothesis consistent with all positive examples.


## 1. Imports

In [1]:
import pandas as pd
import numpy as np


## 2. Load the Dataset

In [2]:
df = pd.read_csv('play_tennis_50.csv')

print("Shape:", df.shape)
df.head(10)


Shape: (50, 5)


,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [3]:
df.info()
print("\nMissing values:\n", df.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Outlook      50 non-null     object
 1   Temperature  50 non-null     object
 2   Humidity     50 non-null     object
 3   Wind         50 non-null     object
 4   PlayTennis   50 non-null     object
dtypes: object(5)
memory usage: 2.1+ KB

Missing values:
 Outlook        0
Temperature    0
Humidity       0
Wind           0
PlayTennis     0
dtype: int64


## 3. Identify Attributes, Target, and Positive Class

The last column is treated as the target (`PlayTennis`). The positive class label is `'Yes'`.


In [4]:
target_col = df.columns[-1]
attribute_cols = df.columns[:-1].tolist()
positive_label = 'Yes'

print("Attributes:", attribute_cols)
print("Target column:", target_col)
print("\nClass distribution:\n", df[target_col].value_counts())


Attributes: ['Outlook', 'Temperature', 'Humidity', 'Wind']
Target column: PlayTennis

Class distribution:
 PlayTennis
Yes    33
No     17
Name: count, dtype: int64


## 4. Extract Positive Examples

In [5]:
positive_examples = df[df[target_col] == positive_label][attribute_cols].reset_index(drop=True)

print(f"Number of positive examples: {len(positive_examples)} out of {len(df)} total rows")
positive_examples.head(10)


Number of positive examples: 33 out of 50 total rows


,Outlook,Temperature,Humidity,Wind
0,Overcast,Hot,High,Weak
1,Rain,Mild,High,Weak
2,Rain,Cool,Normal,Weak
3,Overcast,Cool,Normal,Strong
4,Sunny,Cool,Normal,Weak
5,Rain,Mild,Normal,Weak
6,Sunny,Mild,Normal,Strong
7,Overcast,Mild,High,Strong
8,Overcast,Hot,Normal,Weak
9,Sunny,Hot,Normal,Weak


## 5. Implement the Find-S Algorithm

In [6]:
def find_s(positive_examples: pd.DataFrame):
    """
    Runs the Find-S algorithm over a DataFrame of positive examples only.
    Returns the final hypothesis (list) and the trace of hypotheses after
    processing each positive example (for inspection/explanation).
    """
    attributes = positive_examples.columns.tolist()
    hypothesis = None
    trace = []

    for i, row in positive_examples.iterrows():
        row_values = row.tolist()

        if hypothesis is None:
            # Step 1: initialize hypothesis with the first positive example
            hypothesis = row_values.copy()
        else:
            # Step 2: generalize hypothesis where it disagrees with the example
            hypothesis = [
                h_val if h_val == r_val else '?'
                for h_val, r_val in zip(hypothesis, row_values)
            ]

        trace.append({
            'example_num': i + 1,
            'example': dict(zip(attributes, row_values)),
            'hypothesis_after_this_example': dict(zip(attributes, hypothesis))
        })

    return hypothesis, trace, attributes


final_hypothesis, trace, attributes = find_s(positive_examples)


## 6. Step-by-Step Trace

In [7]:
for step in trace:
    print(f"Example {step['example_num']}: {step['example']}")
    print(f"  -> Hypothesis: {step['hypothesis_after_this_example']}")
    print()


Example 1: {'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak'}
  -> Hypothesis: {'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak'}

Example 2: {'Outlook': 'Rain', 'Temperature': 'Mild', 'Humidity': 'High', 'Wind': 'Weak'}
  -> Hypothesis: {'Outlook': '?', 'Temperature': '?', 'Humidity': 'High', 'Wind': 'Weak'}

Example 3: {'Outlook': 'Rain', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak'}
  -> Hypothesis: {'Outlook': '?', 'Temperature': '?', 'Humidity': '?', 'Wind': 'Weak'}

Example 4: {'Outlook': 'Overcast', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong'}
  -> Hypothesis: {'Outlook': '?', 'Temperature': '?', 'Humidity': '?', 'Wind': '?'}

Example 5: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak'}
  -> Hypothesis: {'Outlook': '?', 'Temperature': '?', 'Humidity': '?', 'Wind': '?'}

Example 6: {'Outlook': 'Rain', 'Temperature': 'Mild', 'Humidity': 'Normal', 'Wind': '

## 7. Final Hypothesis

In [8]:
final_hypothesis_dict = dict(zip(attributes, final_hypothesis))

print("Final Hypothesis (most specific, consistent with all positive examples):\n")
for attr, val in final_hypothesis_dict.items():
    print(f"  {attr:<12} = {val}")

hypothesis_str = "<" + ", ".join(final_hypothesis) + ">"
print("\nCompact form:")
print(f"  h = {hypothesis_str}")


Final Hypothesis (most specific, consistent with all positive examples):

  Outlook      = ?
  Temperature  = ?
  Humidity     = ?
  Wind         = ?

Compact form:
  h = <?, ?, ?, ?>


## 8. Interpretation

- Any attribute in the final hypothesis with value **`?`** means that attribute's value did **not**
  matter across the positive examples seen — Find-S generalized it away.
- Any attribute that still holds a **specific value** means every single positive example agreed
  on that value for that attribute — Find-S kept it specific.
- If the final hypothesis is entirely `<?, ?, ?, ?>`, it means the positive examples were varied
  enough across every attribute that no single value could describe them all — Find-S generalizes
  to the point of admitting any input as positive (this can happen with larger, more diverse
  datasets like this one).

**Key limitation to note:** Find-S ignores negative examples entirely, so it does not verify that
the hypothesis correctly *excludes* the negative examples — it only guarantees consistency with
the positives seen.
